In [8]:
import torch
from torch import nn

# 3.延后初始化

*延后初始化*（defers initialization），
即直到数据第一次通过模型传递时，框架才会动态地推断出每个层的大小。

当使用卷积神经网络时，
由于输入维度（即图像的分辨率）将影响每个后续层的维数，
有了该技术将更加方便。  

现在我们在编写代码时无须知道维度是什么就可以设置参数，
这种能力可以大大简化定义和修改模型的任务。

## 应用场景

延后初始化通常在以下场景中使用：
- 输入数据的形状不确定，延后初始化允许模型根据实际输入动态调整结构，无需提前指定参数形状；
- 对于包含大量参数的大型模型，延后初始化以避免在模型构建时占用过多内存资源，直到真正访问这些参数时才动态分配。
>由于延后初始化，一些潜在的参数配置问题可能要到实际调用时才被暴露出来，而不是模型构建时就可检测到，增加了调试的难度。

接下来，我们将介绍延后初始化机制。

## 3.1 引例-懒汉模式

单例设计中的**懒汉模式** (Lazy Singleton) 与模型参数的延后初始化有相似的延迟操作思想，都能避免提前的内存占用和对象创建。

In [40]:
class Singleton:
    _instance = None
    _initialized = False

    def __new__(cls):
        if cls._instance is None:
            print("创建唯一实例")
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        print('init')
        if not self._initialized:
            # 初始化代码
            self._initialized = True

In [41]:
s1=Singleton()

创建唯一实例
init


In [42]:
s2=Singleton()

init


In [43]:
s1 is s2

True

懒汉模式是实现单例的方式之一，保证类在程序运行期间仅有一个实例，该实例仅在第一次被访问时才创建。  
*注:这里仅为示例, 多线程时不安全*

1. 懒汉模式的实现原理

    - `_instance` 是一个类属性，初始为 `None`，用于存储唯一的实例。

    - `__new__` 是创建实例的静态方法，在 `__init__` 之前被调用。

    - 当第一次调用 `Singleton()` 时，`cls._instance` 为 `None`，进入 `if` 分支，调用 `super().__new__()` 创建实例并赋值给 `cls._instance`。

    - 后续调用直接返回 `cls._instance`，不会再创建新对象。

这种模式将实例化延迟到真正需要时，因此称为“懒汉式”。

2. `__new__` 是一个类方法（尽管没有用 @classmethod 装饰，但它在 Python 中被特殊处理为静态方法，第一个参数自动传入类对象）。如果 `__new__` 返回了 cls 的实例，Python 会自动调用 `__init__` 进行初始化。<br>  
按照约定，类方法的第一个参数命名为 `cls`（代表类本身）

## 3.2 手动实现

In [35]:
class LazyLinear(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = None  # 不在初始化时设置 Linear 层

    def forward(self, data):
        if self.linear is None:
            self.linear = nn.Linear(in_features=data.shape[-1], 
                                    out_features=10)  # 根据输入的形状初始化 Linear 层
        return self.linear(data)

In [36]:
net=LazyLinear()
print(net.linear)

None


In [30]:
X=torch.randn(2,5)
print(X)
print(net.forward(X),'\n')
print(net.linear)

tensor([[ 0.3896,  0.9172, -1.5009,  0.7110, -1.3623],
        [-1.1463, -1.0823, -0.5056,  0.4500,  0.4121]])
tensor([[-1.0129e+00, -2.9013e-02,  5.6402e-02, -3.7766e-01,  2.3611e-01,
         -7.2887e-01, -2.5644e-01,  4.1853e-01,  4.5065e-01, -7.4061e-01],
        [ 4.7058e-02, -5.4900e-01,  2.8182e-01, -2.3161e-01,  5.0323e-01,
          6.9298e-04, -1.8024e-02, -2.3184e-01,  5.0236e-01, -3.0956e-01]],
       grad_fn=<AddmmBackward0>) 

Linear(in_features=5, out_features=10, bias=True)


## 3.3 自动实现

PyTorch 从 1.8 版本开始，向`torch.nn`模块中引入了`LazyModuleMixin`类。`nn.Module`中的层通过继承`LazyModuleMixin`类，以获得延后初始化的特性。  
于是，这些层在接收到数据并前向传播时，自动推断`in_features`参数并初始化。
>这些有延后初始化特性的类以Lazy...为前缀

In [44]:
class LazyLinear_torch(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear=nn.LazyLinear(out_features=3)

    def forward(self,X):
        return self.linear(X)

In [46]:
net=LazyLinear_torch()

In [47]:
net

LazyLinear_torch(
  (linear): LazyLinear(in_features=0, out_features=3, bias=True)
)

In [48]:
net(X)

tensor([[-0.1724,  0.2463, -0.7248],
        [-0.3136,  1.0392,  0.4154]], grad_fn=<AddmmBackward0>)

In [49]:
net

LazyLinear_torch(
  (linear): Linear(in_features=5, out_features=3, bias=True)
)

可以看到`in_features`从 0 变成了 5

*注:这一块还在开发中,API可能会变化*

## 小结

* 延后初始化使框架能够自动推断参数形状，使修改模型架构变得容易，避免了一些常见的错误。
* 我们可以通过模型传递数据，使框架最终初始化参数。

## 练习

1. 如果指定了第一层的输入尺寸，但没有指定后续层的尺寸，会发生什么？是否立即进行初始化？
1. 如果指定了不匹配的维度会发生什么？
1. 如果输入具有不同的维度，需要做什么？提示：查看参数绑定的相关内容。


[Discussions](https://discuss.d2l.ai/t/5770)
